<img src="logo.png" alt="Vegeta" width="240">

# Quadcopter frame — vibration, cyclic loads and life over three mission types

Notebook 08 sized the frame for two static cases. This one asks **how long it lasts**:

```
propeller data (11) ─► excitation lines (1P, 3P, unbalance force)
                                   │
frame + motor masses ─► modal analysis (Talos) ─► natural frequencies ─► Campbell diagram, margins
                                   │
three missions (Chronos) ─► rainflow + amplified vibration cycles ─► load spectrum per mission
                                   │
unit FEA cases (Talos) ─► stress per unit load ─► damage per mission, hotspot map ─► static re-check
                                   │
fleet usage ─► long-term simulation ─► hours to failure
```

Every input is explicit and coarse: lumped masses, one damping ratio, an assumed S-N curve for
PETG-CF, load levels you wrote. The value is in the *comparison* (missions, balance grades, designs)
and in the record; validate the absolute life with a test. Nothing runs unless you run the cell.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz

RUNS = Path("_runs/quad_life"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "quad_frame.py"
shutil.copy(Path("designs/quad_frame.py"), design_file)
frame_design = dedalus.load_design(f"{design_file}:QuadFrame")

# ---- hand-copied inputs (on purpose: the record shows what was assumed) --------------------------
PREFERRED = dict(arm_height=8.0, plate_thickness=8.0)      # notebook 08: preferred revision (taller arms)
PROP = {                                                   # notebook 11: _runs/propeller/quad_5x43.json, points
    "hover":  {"rpm": 8197.0,  "thrust_N": 1.17,  "unbalance_N": 0.11},
    "cruise": {"rpm": 9346.0,  "thrust_N": 1.52,  "unbalance_N": 0.12},
    "full":   {"rpm": 28681.0, "thrust_N": 14.29, "unbalance_N": 0.38},   # current-limited; the frame case uses 8 N
}
BLADES = 3
MOTOR_PROP_MASS_T = 35e-6          # tonnes (35 g motor + prop) on each pad
STACK_MASS_T = 240e-6              # FC + ESC + battery + camera on the stack bolts
MAX_THRUST_N = 8.0                 # per motor, the structural sizing value of notebook 08

## 1. The frame, its masses, and a mesh used by every analysis

The mesh is generated once and copied per load case: the unit cases and the modal analysis must live
on the *same* nodes for the superposition later.

In [ ]:
p = frame_design.resolve(**PREFERRED)
frame = frame_design.generate(**PREFERRED)
cad = frame.export(RUNS / "cad")
PETG_CF = talos.Material("PETG-CF", youngs_modulus=4800.0, poissons_ratio=0.38, density=1.25e-9, yield_strength=45.0,
                         source="nominal filament datasheet, XY orientation")

def frame_regions(p):
    R, m, r = p["wheelbase"] / 2, p["motor_pattern"] / 2, p["motor_hole"] / 2 + 0.2
    regions = []
    for k in range(4):
        a = math.radians(45 + 90 * k)
        cx, cy = R * math.cos(a), R * math.sin(a)
        regions.append(talos.SurfacesInBox(f"motor{k}", (cx - m - r, cy - m - r, -0.1, cx + m + r, cy + m + r, 100.0)))
    s, r = p["stack_pattern"] / 2, p["stack_hole"] / 2 + 0.2
    for k, (sx, sy) in enumerate([(s, s), (-s, s), (s, -s), (-s, -s)]):
        regions.append(talos.SurfacesInBox(f"stack{k}", (sx - r, sy - r, -0.1, sx + r, sy + r, 100.0)))
    return regions

REGIONS = frame_regions(p)
SUPPORTS = [talos.FixedSupport(f"stack{k}") for k in range(4)]
MASSES = [talos.PointMass(f"motor{k}", MOTOR_PROP_MASS_T) for k in range(4)] + \
         [talos.PointMass(f"stack{k}", STACK_MASS_T / 4) for k in range(4)]
MESH = talos.MeshSettings(element_size=5.0)

def model(loads, name):
    return talos.StructuralModel(cad.artifacts["step"], "mm-N-MPa", PETG_CF, REGIONS, SUPPORTS, loads, MESH, name=name, masses=MASSES)

base = model([], "modal")
mesh_res = base.mesh(RUNS / "mesh", progress=True)
print(mesh_res)

def case_dir(name):                         # a copy of the meshed directory per analysis
    d = RUNS / name
    if not d.exists():
        shutil.copytree(RUNS / "mesh", d)
    return d

## 2. Natural frequencies with the motors and the stack on board

The pads carry 35 g each (motor + propeller), the stack bolts 240 g. Without those masses the arm
frequencies would come out several times too high.

In [ ]:
modes = base.solve_modes(case_dir("modal"), n_modes=8, progress=True)
print(modes)
freqs = modes.metrics["frequencies_hz"]

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=1))     # arm bending (vertical), motor mass at the tip

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=5))     # in-plane arm bending: what the unbalance force excites

### Campbell diagram: where the rotor lines cross the frame modes

1P is the shaft frequency, 3P the blade-pass. Hover and cruise sit well above the arm modes; the
**spool-up** from 0 to hover rpm crosses them (the arms ring for a second on every take-off), and the
frame is only safe because it passes through quickly.

In [ ]:
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03, source="Talos modal, PETG-CF nominal E, lumped masses")
rpm_ops = {k: v["rpm"] for k, v in PROP.items()}
fig = structure.campbell({"1P": 1, "3P": BLADES}, np.linspace(1000, 30000, 30), operating_rpm=rpm_ops)
lines = {k: v["rpm"] / 60 for k, v in PROP.items()}
pd.DataFrame({k: {"1P_hz": f, "nearest_mode_hz": structure.nearest_mode(f), "margin": structure.margin(f),
                  "amplification_1P": float(structure.amplification(f)[0])} for k, f in lines.items()}).round(2)

## 3. Unit load cases — the stress "per newton" of each load pattern

Three patterns, each a normal linear static run at a known load:

| pattern | unit case | level unit |
|---|---|---|
| `thrust` | +8 N up on every motor's bolt holes | N per motor |
| `unbalance` | 1 N horizontal at motor 0 (a rotating unbalance seen by the arm) | N |
| `landing` | +40 N up on motor 0 only | N |

Because the analysis is linear, the stress at any level is level × unit stress — that is what lets
a mission with 10⁵ vibration cycles be evaluated without 10⁵ solves.

In [ ]:
UNIT = {
    "thrust":    (model([talos.Force(f"motor{k}", fz=MAX_THRUST_N) for k in range(4)], "thrust"), MAX_THRUST_N),
    "unbalance": (model([talos.Force("motor0", fx=1.0)], "unbalance"), 1.0),
    "landing":   (model([talos.Force("motor0", fz=40.0)], "landing"), 40.0),
}
unit_results = {}
for name, (m, load) in tqdm(UNIT.items(), desc="unit cases"):
    unit_results[name] = m.solve(case_dir(name), progress=False)
    r = unit_results[name]
    print(f"{name:<10} {'ok' if r.ok else 'FAILED'}  max von Mises {r.metrics.get('max_von_mises', float('nan')):.2f} MPa at {load:g} N")
unit_cases = {k: (unit_results[k], UNIT[k][1]) for k in UNIT}

In [ ]:
tviz.show(tviz.plot_results(unit_results["unbalance"], field="von_mises"))   # 1 N sideways on one pad

## 4. Three missions

Levels are **per-motor thrust in N** (`thrust`), the rotating force in N (`unbalance`) and the one-arm
landing force in N (`landing`). The unbalance is ISO G 6.3 from notebook 11. `repeat` makes a segment
an excursion from the previous level, so every punch-out closes a load cycle.

In [ ]:
def unb(point, rpm=None, force=None):
    rpm = rpm or PROP[point]["rpm"]; force = force if force is not None else PROP[point]["unbalance_N"]
    return chronos.Excitation(f"unbalance {point}", rpm / 60, force, "unbalance")

SPOOL = chronos.Excitation("unbalance spool-up", 4500 / 60, 0.06, "unbalance")     # passes through the arm modes
HOVER, CRUISE = PROP["hover"]["thrust_N"], PROP["cruise"]["thrust_N"]

missions = {
    "inspection": chronos.Mission("inspection", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("take-off", 8, {"thrust": 2.0}, (unb("hover"),)),
        chronos.Segment("hover + slow moves", 660, {"thrust": HOVER}, (unb("hover"),)),
        chronos.Segment("position changes", 4, {"thrust": 1.8}, (unb("cruise"),), repeat=12),
        chronos.Segment("landing", 3, {"thrust": 0.6, "landing": 12.0}),
    ), "12 min structure inspection: mostly hover, gentle moves, soft landing"),
    "freestyle": chronos.Mission("freestyle", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("hover", 60, {"thrust": HOVER}, (unb("hover"),)),
        chronos.Segment("punch-out", 1.5, {"thrust": MAX_THRUST_N}, (unb("full", force=0.38),), repeat=40),
        chronos.Segment("hard turns", 2.0, {"thrust": 4.0}, (unb("cruise", rpm=15000, force=0.20),), repeat=60),
        chronos.Segment("cruise between tricks", 120, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("hard landing", 2, {"thrust": 0.8, "landing": 40.0}),
    ), "5 min freestyle: 40 punch-outs, 60 hard turns, one hard landing"),
    "cruise": chronos.Mission("cruise", (
        chronos.Segment("spool-up", 3, {"thrust": 0.5}, (SPOOL,)),
        chronos.Segment("climb", 20, {"thrust": 2.5}, (unb("cruise"),)),
        chronos.Segment("cruise out", 420, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("gust corrections", 2.0, {"thrust": 2.4}, (unb("cruise"),), repeat=30),
        chronos.Segment("cruise back", 420, {"thrust": CRUISE}, (unb("cruise"),)),
        chronos.Segment("landing", 3, {"thrust": 0.6, "landing": 25.0}),
    ), "15 min tilted cruise out and back with gust corrections"),
}
for m in missions.values():
    fig = m.profile(patterns=["thrust", "landing"])
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "segments": len(m.segments), "patterns": ", ".join(m.patterns)} for k, m in missions.items()}).T

## 5. Load spectra: rainflow for the manoeuvres, amplified vibration for the rotor

`build_spectrum` counts the manoeuvre cycles of each pattern (ASTM rainflow on the level sequence) and
adds one block per segment and excitation: cycles = frequency × time, amplitude = force × dynamic
amplification from the nearest frame mode. Look at the spool-up block: small force, large factor.

In [ ]:
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k}.json")
    fig = sp.plot()
spectra["freestyle"].table().round(4)

## 6. Damage per mission on the whole frame (Talos), and the hotspot

S-N curve for PETG-CF — **assumed**: σ_f = 80 MPa, b = −0.11, Goodman with 55 MPa ultimate. Printed
polymers scatter a lot; a few coupons in the print orientation of the arms would replace these numbers.

In [ ]:
CURVE = talos.FatigueCurve("PETG-CF (assumed)", sigma_f=80.0, b=-0.11, ultimate=55.0,
                           source="assumed Basquin fit; replace with coupon tests")
fatigue = {}
for k in tqdm(missions, desc="fatigue"):
    fatigue[k] = talos.assess_fatigue(unit_cases, spectra[k].to_dict(), CURVE, workdir=RUNS / f"fatigue_{k}")
life = pd.DataFrame({k: {"duration_min": missions[k].duration_h * 60, "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"],
                         "hours_to_failure": f.result.metrics["hours_to_failure"],
                         "hotspot": tuple(round(x, 1) for x in f.result.metrics["hotspot_location"])} for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_results["thrust"].artifacts["mesh"]))

In [ ]:
contrib = pd.Series(fatigue[worst].contributions).sort_values(ascending=False)
ax = contrib.head(8).plot.barh(figsize=(8, 3.2), title=f"{worst}: damage contributions at the hotspot"); ax.invert_yaxis()
contrib.head(8).round(6)

## 7. Back to the structural calculation: the worst combined static state

The spectrum also tells which *combination* of levels is the worst static case in each mission
(maximum thrust with the landing load never coincide; the peak of each pattern does with the vibration
amplitude). By superposition the peak stress at the hotspot is Σ level × unit stress: a static
safety-factor check that includes the vibration — this is the "pass it back to the FEM" step.

In [ ]:
hot = fatigue[worst].hotspot
unit_vm = {k: float(talos.read_frd(r.artifacts["frd"]).von_mises[hot]) / UNIT[k][1] for k, r in unit_results.items()}
rows = {}
for k, sp in spectra.items():
    peak = {}
    for b in sp.blocks:
        peak[b.pattern] = max(peak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
    stress = sum(peak.get(pat, 0.0) * unit_vm[pat] for pat in unit_vm)
    rows[k] = {**{f"peak_{pat}": peak.get(pat, 0.0) for pat in unit_vm}, "hotspot_stress_MPa": stress,
               "SF_yield": PETG_CF.yield_strength / stress}
pd.DataFrame(rows).T.round(2)

## 8. Long-term: a fleet usage over thousands of flights

Damage per mission is combined over a usage mix (drawn at random, seeded) until Miner's sum reaches 1.
Two other mixes and a balanced-propeller variant (G 2.5 instead of G 6.3: unbalance force ÷ 2.5) show
what actually drives the life.

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: m.duration_h for k, m in missions.items()}
usage = {"inspection": 0.6, "freestyle": 0.15, "cruise": 0.25}
sim = chronos.simulate_life(damage, hours, usage, n_flights=40000, seed=0)
fig = sim.plot()
print(f"failure after {sim.flights_to_failure:.0f} flights / {sim.hours_to_failure:.0f} h with usage {usage}")

In [ ]:
def life_hours(dmg, mix):
    return chronos.simulate_life(dmg, hours, mix, n_flights=200000, seed=None).hours_to_failure

balanced = {}
for k, sp in spectra.items():
    sp2 = chronos.LoadSpectrum(sp.mission, sp.duration_s,
                               [chronos.Block(b.pattern, b.mean, b.amplitude / 2.5 if b.pattern == "unbalance" else b.amplitude, b.cycles, b.source) for b in sp.blocks],
                               sp.patterns)
    balanced[k] = talos.assess_fatigue(unit_cases, sp2.to_dict(), CURVE).result.metrics["damage_per_pass"]
mixes = {"inspection-heavy": {"inspection": 0.6, "freestyle": 0.15, "cruise": 0.25},
         "freestyle-heavy":  {"inspection": 0.2, "freestyle": 0.6,  "cruise": 0.2},
         "cruise-only":      {"inspection": 0.0, "freestyle": 0.0,  "cruise": 1.0}}
pd.DataFrame({name: {"hours, G 6.3 props": life_hours(damage, mix), "hours, balanced G 2.5": life_hours(balanced, mix)}
              for name, mix in mixes.items()}).T.round(0)

## 9. Export and record

In [ ]:
summary = {
    "design": {"file": "designs/quad_frame.py", "parameters": p},
    "modes_hz": freqs, "damping_ratio": structure.damping_ratio,
    "excitations_hz": lines, "unit_cases": {k: {"load": UNIT[k][1], "max_von_mises": unit_results[k].metrics["max_von_mises"]} for k in UNIT},
    "curve": CURVE.__dict__, "missions": {k: m.describe() for k, m in missions.items()},
    "damage_per_mission": damage, "hours_per_mission": hours,
    "usage": usage, "hours_to_failure": sim.hours_to_failure, "flights_to_failure": sim.flights_to_failure,
    "spectra": {k: str(RUNS / f"spectrum_{k}.json") for k in missions},
}
(RUNS / "life.json").write_text(json.dumps(summary, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**What to do with this:** the hotspot and the dominant blocks say *where* and *why* the frame will
crack; the balanced-propeller row says whether balancing is worth more than material; the static
re-check says whether the worst mission ever gets near yield. Next revisions (notebook 08's workspace,
or a campaign in notebook 10 with `hours_to_failure` as the objective) close the loop.